# ElasticNet Regression

While Ridge (L2) and Lasso (L1) regression are powerful regularization techniques, they both have distinct limitations in specific situations. **ElasticNet Regression** is a hybrid regularization method that combines the L1 and L2 penalties into a single loss function, leveraging the strengths of both to create a robust model.

---

## 1. Why do we need ElasticNet?

To understand ElasticNet, let's look at the limitations of Ridge and Lasso when dealing with high-dimensional datasets containing correlated features:

1. **Lasso's Arbitrary Selection:** If a group of features is highly correlated (multicollinearity), Lasso tends to randomly select **only one** feature from the group and shrink all other coefficients to exactly zero. This arbitrary selection hurts model interpretability.
2. **Ridge's Feature Inflation:** Ridge retains all features, which is helpful for grouping correlated features. However, if a dataset has $1,000$ columns of noise, Ridge cannot perform feature selection and will keep all $1,000$ noisy columns in the final equation.
3. **Lasso limit on sample size:** If $m > n$ (number of features is greater than number of samples), Lasso can select at most $n$ features before it saturates, leaving the rest at zero.

### ElasticNet Solution:
ElasticNet introduces both penalties. It groups highly correlated features together (like Ridge) and shrinks them collectively, while still driving completely irrelevant noise columns to exactly zero (like Lasso), performing automatic feature selection.

## 2. Mathematical Formulation

ElasticNet adds a weighted combination of L1 and L2 penalties to the Mean Squared Error cost function:

$$L_{\text{ElasticNet}}(\beta) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \cdot \text{l1\_ratio} \sum_{j=1}^{m} |\beta_j| + \frac{\alpha \cdot (1 - \text{l1\_ratio})}{2} \sum_{j=1}^{m} \beta_j^2$$

### Hyperparameters in Scikit-Learn:

1. **`alpha`:** The overall regularization strength. Higher values increase regularization.
2. **`l1_ratio`:** The mixing parameter between L1 and L2 regularization.
   - If **`l1_ratio = 1.0`**, the penalty is strictly L1 (Lasso Regression).
   - If **`l1_ratio = 0.0`**, the penalty is strictly L2 (Ridge Regression).
   - If **`0 < l1_ratio < 1.0`**, the model is a hybrid combination of Ridge and Lasso. The default value is $0.5$ (equal split).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Generate complex high-dimensional dataset with correlated groups
# n_samples = 150, n_features = 50
np.random.seed(42)
n_samples = 150

# Group 1: 5 highly correlated features (true signal 1)
g1 = np.random.normal(0, 1, (n_samples, 1))
X_g1 = g1 + np.random.normal(0, 0.05, (n_samples, 5))

# Group 2: 5 highly correlated features (true signal 2)
g2 = np.random.normal(0, 1, (n_samples, 1))
X_g2 = g2 + np.random.normal(0, 0.05, (n_samples, 5))

# Group 3: 40 noise features (completely uncorrelated random noise)
X_noise = np.random.normal(0, 1, (n_samples, 40))

# Combine features
X_raw = np.hstack([X_g1, X_g2, X_noise])

# Target depends strictly on the group factors g1 and g2
y = 4.0 * g1.ravel() - 3.0 * g2.ravel() + np.random.normal(0, 1.0, n_samples)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"Dataset generated. Shape of X: {X_scaled.shape}")
print("Features 0-4 are Group 1 (correlated true variables).")
print("Features 5-9 are Group 2 (correlated true variables).")
print("Features 10-49 are random noise features.")


In [ ]:
# Fit Ridge, Lasso, and ElasticNet models
ridge = Ridge(alpha=10.0)
lasso = Lasso(alpha=0.15)
enet = ElasticNet(alpha=0.15, l1_ratio=0.5)

ridge.fit(X_scaled, y)
lasso.fit(X_scaled, y)
enet.fit(X_scaled, y)

# Store coefficients in a DataFrame
coef_df = pd.DataFrame({
    'Feature Index': range(50),
    'Ridge Coef (L2)': ridge.coef_,
    'Lasso Coef (L1)': lasso.coef_,
    'ElasticNet Coef (Hybrid)': enet.coef_
})

print("="*75)
print("SUMMARY OF FEATURES KEEPING (NON-ZERO COEFFICIENTS)")
print("="*75)
print(f"Ridge non-zero coefficients:      {np.sum(ridge.coef_ != 0)} / 50")
print(f"Lasso non-zero coefficients:      {np.sum(lasso.coef_ != 0)} / 50")
print(f"ElasticNet non-zero coefficients: {np.sum(enet.coef_ != 0)} / 50")
print("="*75)


In [ ]:
# Plotting the coefficient values for Group 1 (correlated features 0-4)
plt.figure(figsize=(12, 6.5))

x_ticks = np.arange(10)
plt.bar(x_ticks - 0.25, ridge.coef_[:10], width=0.25, color='#2ecc71', label='Ridge (L2)')
plt.bar(x_ticks, lasso.coef_[:10], width=0.25, color='#e74c3c', label='Lasso (L1)')
plt.bar(x_ticks + 0.25, enet.coef_[:10], width=0.25, color='#3498db', label='ElasticNet (L1/L2)')

plt.title('Comparison of Coefficients for Correlated Feature Groups (0-4 and 5-9)', fontsize=12, fontweight='bold')
plt.xlabel('Feature Index')
plt.ylabel('Coefficient Value')
plt.xticks(x_ticks)
plt.legend()
plt.show()

# Print specific coefficients of Group 1
print("Specific coefficients for Group 1 (Indices 0 to 4):")
print(coef_df.iloc[:5].to_string(index=False))
print("\nNotice:")
print("- Ridge splits weights but keeps all noise columns (not shown).")
print("- Lasso zeros out most of the correlated group, arbitrarily selecting only a few features.")
print("- ElasticNet keeps all 5 correlated features in the group and balances their weights, while discarding noise columns.")


## Summary Checklist for ElasticNet Regression

1. **Use on Complex High-Dimensional Data:** ElasticNet is ideal when $m > n$ (more features than samples) or when features exhibit high multicollinearity.
2. **Control Balance using `l1_ratio`:** Use Scikit-Learn's `l1_ratio` hyperparameter (mixing parameter) to balance L1 (sparsity) and L2 (grouping) regularization.
3. **Regularization Scale Sensitivity:** Always standardize variables using `StandardScaler` prior to fitting an ElasticNet model.
4. **Alternative Optimization via SGD:** For very large streaming datasets, implement ElasticNet via `SGDRegressor(penalty='elasticnet')` to train iteratively with gradient descent.